# Ingesta de datos de recursos hídricos de Sonora

Esta libreta de Jupyter se utilizará para la ingesta y validación de datos de almacenamiento de agua en presas del estado de Sonora, con base en el archivo público `hidrico_sonora_2020-actualidad2024.xlsx`.

## Ingesta de datos

### Librerias necesarias

In [1]:
!pip install pydantic


[notice] A new release of pip is available: 25.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from loguru import logger
from tqdm import tqdm
from pydantic import BaseModel, Field, ValidationError, field_validator, ConfigDict
from typing import Optional

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

RAW_DATA_DIR = Path("../data/raw")

HIDRICO_SONORA_URL = (
    "https://datos.sonora.gob.mx/dataset/ee8f639f-5e89-46ae-93b9-934b05fc6233/"
    "resource/213a4f46-7cd1-412f-aeec-4fab8193f51f/download/"
    "hidrico_sonora_2020-actualidad2024.xlsx"
)

### Descarga de datos

In [3]:
def download_hidrico_data():
    """Descarga el archivo de recursos hídricos (presas) de Sonora."""
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

    dest_file = RAW_DATA_DIR / "hidrico_sonora_2020-actualidad2024.xlsx"

    for _ in tqdm([1], total=1):
        try:
            response = requests.get(HIDRICO_SONORA_URL, timeout=30)
            response.raise_for_status()

            with open(dest_file, "wb") as f:
                f.write(response.content)

            logger.success(f"Archivo descargado con éxito: {dest_file}")
        except requests.exceptions.RequestException as e:
            logger.warning(f"Failed to download hidrico data: {e}")

    print("Carpeta destino:", RAW_DATA_DIR)
    print("Existe:", dest_file.exists())

In [4]:
download_hidrico_data()

100%|██████████| 1/1 [00:01<00:00,  1.24s/it]

Carpeta destino: ../data/raw
Existe: True


### Carga de datos descargados a DataFrame

In [5]:
hidrico_sonora_df = pd.read_excel(RAW_DATA_DIR / "hidrico_sonora_2020-actualidad2024.xlsx")
logger.info("Loaded hidrico_sonora_2020-actualidad2024.xlsx")

2026-09-28 21:00:08.610 | INFO     | __main__:<module>:2 - Loaded hidrico_sonora_2020-actualidad2024.xlsx


### Exploración inicial del DataFrame

In [6]:
hidrico_sonora_df.head()

,Clave,Fecha,Almacenamiento(hmÂ³)
0,LCDSO,2020-01-01,738.54
1,PECSO,2020-01-01,2283.69
2,AOBSO,2020-01-01,1683.92
3,AGZCH,2020-01-01,79.10
4,ARCSO,2020-01-01,501.46


In [7]:
hidrico_sonora_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16059 entries, 0 to 16058
Data columns (total 3 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   Clave                  16059 non-null  str           
 1   Fecha                  16059 non-null  datetime64[us]
 2    Almacenamiento(hmÂ³)  16059 non-null  float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 376.5 KB


In [8]:
hidrico_sonora_df.describe()

,Fecha,Almacenamiento(hmÂ³)
count,16059,16059.000000
mean,2022-05-31 11:11:42.783486,360.856447
min,2020-01-01 00:00:00,0.000000
25%,2021-03-22 00:00:00,5.375000
50%,2022-06-11 00:00:00,34.520000
75%,2023-08-15 00:00:00,519.020000
max,2024-09-19 00:00:00,2823.350000
std,NaN,568.345837


## Validación de la descarga de datos de recursos hídricos de Sonora

### Schema basandonos en el diccionario de datos de recursos hídricos de Sonora

El schema `HidricoSonoraSchema` ya se encuentra definido en el archivo `schemas.py`, por lo que podemos importarlo directamente desde allí.

In [9]:
from sonora_agriclimate_eda.schemas import HidricoSonoraSchema

2026-09-28 21:00:08.664 | INFO     | sonora_agriclimate_eda.config:<module>:11 - PROJ_ROOT path is: /Users/luishernandez/Desktop/MCD/2026-1/Ingenieria de caracteristicas/Proyectos/sonora-agriclimate-eda


In [10]:
HidricoSonoraSchema.model_fields

{'Clave': FieldInfo(annotation=Union[str, NoneType], required=False, default=None),
 'Fecha': FieldInfo(annotation=Union[datetime, NoneType], required=False, default=None),
 'almacenamiento_hm3': FieldInfo(annotation=Union[float, NoneType], required=False, default=None, alias='Almacenamiento(hm³)', alias_priority=2, metadata=[Ge(ge=0)])}

### Función para validar los datos descargados de recursos hídricos de Sonora

In [11]:
def validate_df(df: pd.DataFrame) -> pd.DataFrame:
    validated_rows = []

    for index, row in df.iterrows():
        try:
            hidrico = HidricoSonoraSchema.model_validate(row.to_dict())
            validated_rows.append(hidrico.model_dump())

        except ValidationError as e:
            print(f"Error en fila {index}:")
            print(e)

    return pd.DataFrame(validated_rows)

In [12]:
df_validated = validate_df(hidrico_sonora_df)

print(f"Filas totales: {len(hidrico_sonora_df)}")
print(f"Filas válidas: {len(df_validated)}")
print(f"Filas inválidas: {len(hidrico_sonora_df) - len(df_validated)}")

Filas totales: 16059
Filas válidas: 16059
Filas inválidas: 0
